Your task is to create a bert-base-classifier of vacancy areas based on their titles.

Each vacancy can have more than one area so it's **Multi-label classification** not Multiclass classification




In [11]:
import pandas as pd
import numpy as np
import os
from sklearn.metrics import classification_report
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from nltk.tokenize import word_tokenize
from string import punctuation
from tqdm import tqdm

In [12]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [13]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, RandomSampler, Dataset, SequentialSampler
import random
import transformers

# Try two or more different bert-like models(different berts, robertas, debertas, modernberts etc. or any other transformer based model) (**2 points max**)
 your notebook should contain the training process of all your models!

Note that the interfaces of different models may differ slightly.

# ***Model 1***

In [14]:
MODEL_NAME = 'bert-base-uncased'
MAX_SEQ_LENGTH = 64
RESULT_MODEL_PATH = './model.pt'

In [15]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed = 12
seed_everything(seed)

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [17]:
device

device(type='cuda')

In [18]:
punctuation = set('!"$%&\'()*,-/:;<=>?@[\\]^_`{|}~') # убрал #

In [19]:
def clean(text):
    return ' '.join([token.lower() for token in word_tokenize(text) if token not in punctuation])

In [20]:
df = pd.read_csv('./dataset_2020.csv')
df.shape

(78909, 2)

Each vacancy can have more than one area separated be space

Exapmle:

Malware Analyst for Imunify Security,analyst it_security

In [21]:
df_train, df_test = train_test_split(df, train_size=0.9, random_state=42)
df_train, df_valid = train_test_split(df_train, train_size=0.8, random_state=42)

# Finish TextClassificationDataset (**1 point max**)

In [22]:
class TextClassificationDataset(Dataset):
    def __init__(self, data, tokenizer, bianizer):
        self.data = data
        self.tokenizer = tokenizer
        self.sentences = [clean(sent) for sent in data.title.tolist()]
        self.target = [labels.split() for labels in data.area.tolist()]
        self.bianiezer = bianizer
        self.target_one_hot = torch.tensor(
            self.bianiezer.transform(self.target),
            dtype=torch.float
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]

        encoding = self.tokenizer(
            sentence,
            max_length=MAX_SEQ_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        target = self.target_one_hot[idx]

        return input_ids, attention_mask, target


In [23]:
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
binarizer = MultiLabelBinarizer()
labels_train = [labels.split() for labels in df_train.area.tolist()]
binarizer.fit(labels_train)


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

MultiLabelBinarizer()

In [24]:
batch_size = 16

train_dataset = TextClassificationDataset(df_train, tokenizer, binarizer)
train_sampler = RandomSampler(train_dataset)
train_dataloader =  DataLoader(train_dataset, sampler=train_sampler, batch_size=batch_size,)

valid_dataset = TextClassificationDataset(df_valid, tokenizer, binarizer)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size)

test_dataset = TextClassificationDataset(df_test, tokenizer, binarizer)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)


In [25]:
class BertForMultilabel(nn.Module):
    def __init__(self, num_labels: int):
        super().__init__()

        self.bert = transformers.AutoModel.from_pretrained(MODEL_NAME)

        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_labels
        )

    def train_bert(self, train_bert_flag=True):
        for param in self.bert.parameters():
            param.requires_grad = train_bert_flag

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        return logits


In [26]:
num_labels = len(binarizer.classes_)
model = BertForMultilabel(num_labels)
model.to(device)
;

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


''

# Train your classifier with freezed bert and save model with the lowest val loss during training (**2 points max**)

print train/val loss after each epoch


In [27]:
def train(model, iterator, optimizer, criterion):
    model.train()
    epoch_loss = 0.0

    for batch in iterator:
        input_ids, attention_mask, targets = batch

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(logits, targets)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [28]:
def validate(model, iterator, criterion, return_loss=False):
    model.eval()

    epoch_loss = 0.0
    all_preds = []

    with torch.no_grad():
        for batch in iterator:
            input_ids, attention_mask, targets = batch

            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            targets = targets.to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, targets)
            epoch_loss += loss.item()

            batch_preds = logits_to_labels(logits)
            all_preds.extend(batch_preds)

    avg_loss = epoch_loss / len(iterator)

    if return_loss:
        return avg_loss, all_preds

    return all_preds

In [29]:
def logits_to_labels(logits):
    preds = nn.Sigmoid()(logits.view(-1, num_labels))
    preds = preds.to('cpu').numpy()>0.5
    return preds.tolist()

In [30]:
model.train_bert(False)

In [31]:
epochs = 3

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=1,
    gamma=0.5
)

In [32]:
best_val_loss = float('inf')

for epoch in range(epochs):
    train_loss = train(
        model,
        train_dataloader,
        optimizer,
        criterion
    )

    val_loss, _ = validate(
        model,
        valid_dataloader,
        criterion,
        return_loss=True
    )

    print(
        f'Epoch {epoch + 1}/{epochs} | '
        f'Train loss: {train_loss:.4f} | '
        f'Val loss: {val_loss:.4f}'
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), RESULT_MODEL_PATH)

    scheduler.step()

Epoch 1/3 | Train loss: 0.0469 | Val loss: 0.0223
Epoch 2/3 | Train loss: 0.0299 | Val loss: 0.0189
Epoch 3/3 | Train loss: 0.0279 | Val loss: 0.0176


In [33]:
model.load_state_dict(torch.load(RESULT_MODEL_PATH, map_location=torch.device(device), weights_only=True))
test_preds = validate(model, test_dataloader, criterion)

In [34]:
print(classification_report(binarizer.transform(test_dataset.target), test_preds,
                            target_names=binarizer.classes_))

                 precision    recall  f1-score   support

          admin       1.00      0.54      0.70        61
        analyst       0.97      0.93      0.95       302
    architector       1.00      0.62      0.77       111
      assistant       1.00      0.21      0.35        14
     consultant       1.00      0.35      0.52        23
          coord       1.00      0.09      0.17        11
  data_engineer       0.97      0.51      0.67       136
 data_scientist       0.98      0.84      0.91       154
       designer       0.98      0.88      0.93       409
devel_metodolog       1.00      0.93      0.96        44
         devops       1.00      0.96      0.98       338
       director       0.88      0.41      0.56        17
     doc_writer       1.00      0.89      0.94        18
    it_security       1.00      0.26      0.41        54
machine_learner       0.94      0.40      0.57        42
        manager       0.83      0.67      0.74       427
       networks       1.00    

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Train your classifier with unfreezed bert and save model with the lowest val loss during training (**2 points max**)

print train/val loss after each epoch

In [35]:
epochs = 3
lr = 2e-5

WARMUP_PROPORTION = 0.1
warmup_steps = int(
    len(train_dataloader) * epochs * WARMUP_PROPORTION
)

In [36]:
model.train_bert(True)

In [37]:
t_total = len(train_dataloader) * epochs

no_decay = ['bias', 'LayerNorm.weight']

param_optimizer = list(model.named_parameters())

optimizer_grouped_parameters = [
    {
        'params': [
            p for n, p in param_optimizer
            if not any(nd in n for nd in no_decay)
        ],
        'weight_decay': 0.001
    },
    {
        'params': [
            p for n, p in param_optimizer
            if any(nd in n for nd in no_decay)
        ],
        'weight_decay': 0.0
    },
]

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    optimizer_grouped_parameters,
    lr=lr
)

scheduler = transformers.get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=t_total
)

In [38]:
best_val_loss = float('inf')

for epoch in range(epochs):
    train_loss = train(
        model,
        train_dataloader,
        optimizer,
        criterion
    )

    val_loss, _ = validate(
        model,
        valid_dataloader,
        criterion,
        return_loss=True
    )

    print(
        f'Epoch {epoch + 1}/{epochs} | '
        f'Train loss: {train_loss:.4f} | '
        f'Val loss: {val_loss:.4f}'
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), RESULT_MODEL_PATH)

    scheduler.step()

Epoch 1/3 | Train loss: 0.0275 | Val loss: 0.0176
Epoch 2/3 | Train loss: 0.0249 | Val loss: 0.0148
Epoch 3/3 | Train loss: 0.0214 | Val loss: 0.0125


In [39]:
model.load_state_dict(torch.load(RESULT_MODEL_PATH, map_location=torch.device(device), weights_only=True))
test_preds = validate(model, test_dataloader, criterion)

In [40]:
print(classification_report(binarizer.transform(test_dataset.target), test_preds,
                            target_names=binarizer.classes_))

                 precision    recall  f1-score   support

          admin       0.98      0.80      0.88        61
        analyst       0.98      0.96      0.97       302
    architector       1.00      0.81      0.90       111
      assistant       1.00      0.36      0.53        14
     consultant       0.82      0.39      0.53        23
          coord       1.00      0.64      0.78        11
  data_engineer       0.98      0.64      0.77       136
 data_scientist       0.97      0.92      0.95       154
       designer       0.97      0.95      0.96       409
devel_metodolog       1.00      0.93      0.96        44
         devops       0.99      0.99      0.99       338
       director       0.90      0.53      0.67        17
     doc_writer       1.00      0.89      0.94        18
    it_security       0.95      0.70      0.81        54
machine_learner       0.91      0.71      0.80        42
        manager       0.83      0.81      0.82       427
       networks       1.00    

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# ***Model 2***

In [41]:
MODEL_NAME = 'roberta-base'
MAX_SEQ_LENGTH = 64
RESULT_MODEL_PATH = './roberta_model.pt'

In [42]:
def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed = 12
seed_everything(seed)

In [43]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [44]:
device

device(type='cuda')

In [45]:
punctuation = set('!"$%&\'()*,-/:;<=>?@[\\]^_`{|}~') # убрал #

In [46]:
def clean(text):
    return ' '.join([token.lower() for token in word_tokenize(text) if token not in punctuation])

In [47]:
df = pd.read_csv('./dataset_2020.csv')
df.shape

(78909, 2)

In [48]:
df_train, df_test = train_test_split(df, train_size=0.9, random_state=42)
df_train, df_valid = train_test_split(df_train, train_size=0.8, random_state=42)

In [49]:
class TextClassificationDataset(Dataset):
    def __init__(self, data, tokenizer, bianizer):
        self.data = data
        self.tokenizer = tokenizer
        self.sentences = [clean(sent) for sent in data.title.tolist()]
        self.target = [labels.split() for labels in data.area.tolist()]
        self.bianiezer = bianizer
        self.target_one_hot = torch.tensor(
            self.bianiezer.transform(self.target),
            dtype=torch.float
        )

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sentence = self.sentences[idx]

        encoding = self.tokenizer(
            sentence,
            max_length=MAX_SEQ_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].squeeze(0)
        attention_mask = encoding['attention_mask'].squeeze(0)
        target = self.target_one_hot[idx]

        return input_ids, attention_mask, target


In [50]:
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
binarizer = MultiLabelBinarizer()
labels_train = [labels.split() for labels in df_train.area.tolist()]
binarizer.fit(labels_train)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

MultiLabelBinarizer()

In [51]:
batch_size = 16

train_dataset = TextClassificationDataset(df_train, tokenizer, binarizer)
train_sampler = RandomSampler(train_dataset)
train_dataloader =  DataLoader(train_dataset, sampler=train_sampler, batch_size=batch_size,)

valid_dataset = TextClassificationDataset(df_valid, tokenizer, binarizer)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size)

test_dataset = TextClassificationDataset(df_test, tokenizer, binarizer)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)


In [52]:
class BertForMultilabel(nn.Module):
    def __init__(self, num_labels: int):
        super().__init__()

        self.bert = transformers.AutoModel.from_pretrained(MODEL_NAME)

        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_labels
        )

    def train_bert(self, train_bert_flag=True):
        for param in self.bert.parameters():
            param.requires_grad = train_bert_flag

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        return logits


In [53]:
num_labels = len(binarizer.classes_)
model = BertForMultilabel(num_labels)
model.to(device)
;

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


''

# С заморозкой

In [54]:
def train(model, iterator, optimizer, criterion):
    model.train()
    epoch_loss = 0.0

    for batch in iterator:
        input_ids, attention_mask, targets = batch

        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(logits, targets)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [55]:
def validate(model, iterator, criterion, return_loss=False):
    model.eval()

    epoch_loss = 0.0
    all_preds = []

    with torch.no_grad():
        for batch in iterator:
            input_ids, attention_mask, targets = batch

            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            targets = targets.to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = criterion(logits, targets)
            epoch_loss += loss.item()

            batch_preds = logits_to_labels(logits)
            all_preds.extend(batch_preds)

    avg_loss = epoch_loss / len(iterator)

    if return_loss:
        return avg_loss, all_preds

    return all_preds

In [56]:
def logits_to_labels(logits):
    preds = nn.Sigmoid()(logits.view(-1, num_labels))
    preds = preds.to('cpu').numpy()>0.5
    return preds.tolist()

In [57]:
model.train_bert(False)

In [58]:
epochs = 3

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=1,
    gamma=0.5
)

In [59]:
best_val_loss = float('inf')

for epoch in range(epochs):
    train_loss = train(
        model,
        train_dataloader,
        optimizer,
        criterion
    )

    val_loss, _ = validate(
        model,
        valid_dataloader,
        criterion,
        return_loss=True
    )

    print(
        f'Epoch {epoch + 1}/{epochs} | '
        f'Train loss: {train_loss:.4f} | '
        f'Val loss: {val_loss:.4f}'
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), RESULT_MODEL_PATH)

    scheduler.step()

Epoch 1/3 | Train loss: 0.1022 | Val loss: 0.0802
Epoch 2/3 | Train loss: 0.0781 | Val loss: 0.0719
Epoch 3/3 | Train loss: 0.0731 | Val loss: 0.0686


In [60]:
model.load_state_dict(torch.load(RESULT_MODEL_PATH, map_location=torch.device(device), weights_only=True))
test_preds = validate(model, test_dataloader, criterion)

In [61]:
print(classification_report(binarizer.transform(test_dataset.target), test_preds,
                            target_names=binarizer.classes_))

                 precision    recall  f1-score   support

          admin       0.00      0.00      0.00        61
        analyst       1.00      0.00      0.01       302
    architector       0.00      0.00      0.00       111
      assistant       0.00      0.00      0.00        14
     consultant       0.00      0.00      0.00        23
          coord       0.00      0.00      0.00        11
  data_engineer       0.00      0.00      0.00       136
 data_scientist       0.00      0.00      0.00       154
       designer       0.00      0.00      0.00       409
devel_metodolog       0.00      0.00      0.00        44
         devops       1.00      0.41      0.59       338
       director       0.00      0.00      0.00        17
     doc_writer       0.00      0.00      0.00        18
    it_security       0.00      0.00      0.00        54
machine_learner       0.00      0.00      0.00        42
        manager       0.00      0.00      0.00       427
       networks       0.00    

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Без заморозки

In [62]:
epochs = 3
lr = 2e-5

WARMUP_PROPORTION = 0.1
warmup_steps = int(
    len(train_dataloader) * epochs * WARMUP_PROPORTION
)

In [63]:
model.train_bert(True)

In [64]:
t_total = len(train_dataloader) * epochs

no_decay = ['bias', 'LayerNorm.weight']

param_optimizer = list(model.named_parameters())

optimizer_grouped_parameters = [
    {
        'params': [
            p for n, p in param_optimizer
            if not any(nd in n for nd in no_decay)
        ],
        'weight_decay': 0.001
    },
    {
        'params': [
            p for n, p in param_optimizer
            if any(nd in n for nd in no_decay)
        ],
        'weight_decay': 0.0
    },
]

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    optimizer_grouped_parameters,
    lr=lr
)

scheduler = transformers.get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=t_total
)

In [65]:
best_val_loss = float('inf')

for epoch in range(epochs):
    train_loss = train(
        model,
        train_dataloader,
        optimizer,
        criterion
    )

    val_loss, _ = validate(
        model,
        valid_dataloader,
        criterion,
        return_loss=True
    )

    print(
        f'Epoch {epoch + 1}/{epochs} | '
        f'Train loss: {train_loss:.4f} | '
        f'Val loss: {val_loss:.4f}'
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), RESULT_MODEL_PATH)

    scheduler.step()

Epoch 1/3 | Train loss: 0.0719 | Val loss: 0.0686
Epoch 2/3 | Train loss: 0.0595 | Val loss: 0.0448
Epoch 3/3 | Train loss: 0.0446 | Val loss: 0.0323


In [66]:
model.load_state_dict(torch.load(RESULT_MODEL_PATH, map_location=torch.device(device), weights_only=True))
test_preds = validate(model, test_dataloader, criterion)

In [67]:
print(classification_report(binarizer.transform(test_dataset.target), test_preds,
                            target_names=binarizer.classes_))

                 precision    recall  f1-score   support

          admin       0.00      0.00      0.00        61
        analyst       0.99      0.88      0.93       302
    architector       1.00      0.50      0.67       111
      assistant       0.00      0.00      0.00        14
     consultant       0.00      0.00      0.00        23
          coord       0.00      0.00      0.00        11
  data_engineer       0.00      0.00      0.00       136
 data_scientist       0.97      0.48      0.64       154
       designer       0.99      0.89      0.94       409
devel_metodolog       0.00      0.00      0.00        44
         devops       0.99      0.99      0.99       338
       director       0.00      0.00      0.00        17
     doc_writer       0.00      0.00      0.00        18
    it_security       0.00      0.00      0.00        54
machine_learner       0.00      0.00      0.00        42
        manager       0.61      0.52      0.56       427
       networks       0.00    

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# Results (3 points max)

Write your conclusion

What models and what training parameters did you use?

What was the reason for your choice?

What were the results?

What metrics do you consider the most important?

| Модель | Micro F1 | Macro F1 | Weighted F1 |
|---|---:|---:|---:|
| BERT, замороженный | 0.91 | 0.70 | 0.90 |
| BERT, дообученный | 0.94 | 0.83 | 0.94 |
| RoBERTa, замороженный | 0.64 | 0.07 | 0.51 |
| RoBERTa, дообученный | 0.85 | 0.35 | 0.81 |

## Заключение

В ходе эксперимента я сравнила две предварительно обученные трансформерные модели — **BERT и RoBERTa** — для многометочной классификации названий вакансий. Для каждой модели я проверила два подхода к обучению: с замороженным энкодером и с дообучением всей модели.

Обучение каждой модели состояло из двух этапов. На первом этапе энкодер оставался замороженным, а классификационная голова обучалась в течение 3 эпох с оптимизатором Adam. Также использовался планировщик скорости обучения StepLR. На втором этапе энкодер размораживался, и вся модель дообучалась ещё 3 эпохи с оптимизатором AdamW. Для этого этапа использовалось линейное расписание скорости обучения с прогревом (linear warmup). В классификационной голове применялся Dropout со значением 0,3.

Я выбрала BERT и RoBERTa, поскольку это предварительно обученные языковые модели, способные учитывать контекст слов. Они подходят для классификации коротких текстов, таких как названия вакансий. Сравнение замороженного и дообученного вариантов позволило оценить, насколько адаптация моделей к конкретной задаче улучшает качество классификации.

Результаты показали, что **дообучение улучшило качество обеих моделей**. Лучший результат продемонстрировал дообученный BERT: Micro F1 составила 0,94, Macro F1 — 0,83, Weighted F1 — 0,94. Дообученная RoBERTa получила значения 0,85, 0,35 и 0,81 соответственно. Замороженный BERT также показал хорошие результаты: Micro F1 составила 0,91. Замороженная RoBERTa оказалась значительно слабее: Micro F1 — 0,64, Macro F1 — всего 0,07.

Наиболее важными метриками для этой задачи я считаю **Micro F1 и Macro F1**. Micro F1 отражает общее качество классификации с учётом всех предсказаний и позволяет оценить эффективность модели в целом. Macro F1 рассчитывается отдельно для каждого класса, после чего значения усредняются с одинаковым весом для всех классов. Поэтому эта метрика особенно важна для набора данных, в котором категории представлены неодинаковым количеством примеров: она позволяет заметить низкое качество на редких классах. Weighted F1 также полезна, поскольку учитывает частоту классов, но может скрывать проблемы с классификацией редких категорий.

Таким образом, **дообученный BERT оказался лучшей моделью среди четырёх рассмотренных вариантов**. При этом разница между Micro F1 (0,94) и Macro F1 (0,83) показывает, что качество классификации отдельных категорий всё ещё различается. В дальнейшем можно сосредоточиться на улучшении предсказаний для редких классов, чтобы повысить качество модели не только в целом, но и для каждой категории.